In [ ]:
import os
import cv2
import urllib.request
import numpy as np
import time
import mujoco
import mujoco.viewer
import mujoco_menagerie as mm
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 0. Download hand_landmarker.task model automatically if missing
model_file = 'hand_landmarker.task'
if not os.path.exists(model_file):
    print("Downloading MediaPipe hand_landmarker.task model file...")
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    urllib.request.urlretrieve(url, model_file)
    print("Download complete!")

# 1. Load official 5-Finger Shadow Hand model
model_path = str(mm.get("shadow_hand").path() / "scene_right.xml")
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

mujoco.mj_resetData(model, data)

# 2. Rotate base forearm (rh_forearm) upright on Z-axis
body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "rh_forearm")
if body_id != -1:
    # Rotate 90 degrees around X-axis so it stands upright on the floor
    model.body_quat[body_id] = [0.7071, 0.7071, 0, 0]
    model.body_pos[body_id] = [0.0, 0.0, 0.1]

mujoco.mj_forward(model, data)

# Shared control targets across thread callbacks for all 5 fingers
latest_ctrl = np.zeros(5)  # [Index, Middle, Ring, Pinky, Thumb]
latest_landmarks = None

def calculate_flexion(landmarks, tip_idx, mcp_idx, wrist_idx=0):
    wrist = np.array([landmarks[wrist_idx].x, landmarks[wrist_idx].y])
    mcp = np.array([landmarks[mcp_idx].x, landmarks[mcp_idx].y])
    tip = np.array([landmarks[tip_idx].x, landmarks[tip_idx].y])
    
    d_mcp = np.linalg.norm(mcp - wrist)
    d_tip = np.linalg.norm(tip - wrist)
    
    return np.clip(1.0 - (d_tip / (d_mcp * 1.8 + 1e-6)), 0.0, 1.0)

# Asynchronous callback for fast multi-threaded processing
def async_result_callback(result: vision.HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    global latest_ctrl, latest_landmarks
    if result.hand_landmarks:
        landmarks = result.hand_landmarks[0]
        latest_landmarks = landmarks
        
        # Flexion ratio for all 5 fingers
        index_flex  = calculate_flexion(landmarks, 8, 5)   # Index
        middle_flex = calculate_flexion(landmarks, 12, 9)  # Middle
        ring_flex   = calculate_flexion(landmarks, 16, 13) # Ring
        pinky_flex  = calculate_flexion(landmarks, 20, 17) # Pinky
        thumb_flex  = calculate_flexion(landmarks, 4, 2)   # Thumb
        
        latest_ctrl = np.array([index_flex, middle_flex, ring_flex, pinky_flex, thumb_flex])

# 3. Setup Async Live-Stream MediaPipe Detector
base_options = python.BaseOptions(model_asset_path=model_file)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    num_hands=1,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    result_callback=async_result_callback
)
detector = vision.HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
    (0,1), (1,2), (2,3), (3,4),          # Thumb
    (0,5), (5,6), (6,7), (7,8),          # Index
    (9,10), (10,11), (11,12),            # Middle
    (13,14), (14,15), (15,16),          # Ring
    (0,17), (17,18), (18,19), (19,20),   # Pinky
    (5,9), (9,13), (13,17)              # Palm base
]

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Minimal buffer for zero camera latency

start_time = time.time()

# 4. Interactive loop running asynchronously
with mujoco.viewer.launch_passive(model, data) as viewer:
    print("5-Finger Real-Time Bionic Tracking Active. Press 'q' on camera window to stop.")
    
    while viewer.is_running() and cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        image = cv2.flip(image, 1)
        h, w, _ = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Dispatch frame asynchronously without blocking loop execution
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        timestamp_ms = int((time.time() - start_time) * 1000)
        detector.detect_async(mp_image, timestamp_ms)

        # Draw tracking skeleton on camera window
        if latest_landmarks is not None:
            for conn in HAND_CONNECTIONS:
                pt1 = (int(latest_landmarks[conn[0]].x * w), int(latest_landmarks[conn[0]].y * h))
                pt2 = (int(latest_landmarks[conn[1]].x * w), int(latest_landmarks[conn[1]].y * h))
                cv2.line(image, pt1, pt2, (0, 255, 0), 2)
                
            for lm in latest_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(image, (cx, cy), 3, (0, 0, 255), -1)

        # Route control signals to 5-finger Shadow Hand actuators
        for i in range(model.nu):
            actuator_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or ""
            if "FF" in actuator_name:       # Index finger
                data.ctrl[i] = latest_ctrl[0] * 1.4
            elif "MF" in actuator_name:     # Middle finger
                data.ctrl[i] = latest_ctrl[1] * 1.4
            elif "RF" in actuator_name:     # Ring finger
                data.ctrl[i] = latest_ctrl[2] * 1.4
            elif "LF" in actuator_name:     # Pinky finger
                data.ctrl[i] = latest_ctrl[3] * 1.4
            elif "TH" in actuator_name:     # Thumb
                data.ctrl[i] = latest_ctrl[4] * 1.2

        # Step physics sub-steps for smooth 60+ FPS motion
        for _ in range(5):
            mujoco.mj_step(model, data)
            
        viewer.sync()

        cv2.imshow('5-Finger Bionic Tracking (Press Q to stop)', image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Real-time tracking active! Press 'q' on camera window to stop.
